In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. MODEL LOADING (Gemma 4 E2B-IT)
# ══════════════════════════════════════════════════════════════════════════════

import os
import torch
# IMPORT AutoModelForCausalLM (or AutoModel) instead of Gemma3ForCausalLM
# IMPORT AutoProcessor for multimodal inputs
from transformers import AutoTokenizer, AutoProcessor, AutoModelForCausalLM 

# Optimize Torch for CPU
torch.set_num_threads(os.cpu_count())
torch.set_num_interop_threads(min(os.cpu_count(), 4))

print(f"PyTorch version: {torch.__version__}")

# Update this path to your new local Gemma 4 folder
GEMMA_PATH = r"C:\Users\divyam.tewary\OneDrive - InterGlobe Aviation Limited\Desktop\kaggle_models\gemma-4-transformers-gemma-4-e2b-it-v1"

print(f"Loading model from: {GEMMA_PATH}...")

try:
    # 1. Try to load a Processor (required for Audio/Vision in E2B)
    tokenizer = AutoTokenizer.from_pretrained(GEMMA_PATH)
    # If it fails, fallback to Tokenizer
    try:
        processor = AutoProcessor.from_pretrained(GEMMA_PATH, trust_remote_code=True)
        print("✅ Processor loaded (Multimodal support active)")
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(GEMMA_PATH, trust_remote_code=True)
        print("✅ Tokenizer loaded (Text-only fallback)")

    # 2. Use AutoModelForCausalLM to dynamically find the right architecture
    model = AutoModelForCausalLM.from_pretrained(
        GEMMA_PATH,
        torch_dtype=torch.bfloat16,    
        device_map="auto",            
        trust_remote_code=True        # CRITICAL for 1-day-old models!
    )
    
    print("✅ Model loaded successfully!")
    print(f"  Device: {model.device}")
    print(f"  Dtype: {model.dtype}")

except Exception as e:
    print(f"❌ Failed to load model. Error: {e}")
    print("Make sure you ran: pip install --upgrade transformers")

C:\Users\divyam.tewary\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.10.0+cpu
Loading model from: C:\Users\divyam.tewary\OneDrive - InterGlobe Aviation Limited\Desktop\kaggle_models\gemma-4-transformers-gemma-4-e2b-it-v1...


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Processor loaded (Multimodal support active)


Loading weights: 100%|██████████| 2011/2011 [00:00<00:00, 4732.70it/s]

✅ Model loaded successfully!
  Device: cpu
  Dtype: torch.bfloat16


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# 1.5. VOCABULARY SURGERY (English-Only Constraint)
# ══════════════════════════════════════════════════════════════════════════════
# Gemma has a 256k vocabulary containing many languages and scripts.
# To prevent "script-switching" hallucinations (like outputting Chinese characters),
# we create a LogitsProcessor that forces the model's probability for non-English 
# tokens to negative infinity.

import string
import torch
from transformers import LogitsProcessor, LogitsProcessorList

print("Scanning vocabulary for non-English tokens...")

# SentencePiece uses ' ' (U+2581) to represent spaces.
allowed_chars = set(string.printable + ' \u2581')
banned_token_ids = []

vocab = tokenizer.get_vocab()
for token, token_id in vocab.items():
    # Always allow special tokens (<bos>, <eos>, <pad>, etc.)
    if token_id in tokenizer.all_special_ids:
        continue
        
    # If the token contains any character outside standard ASCII + SP space
    if any(c not in allowed_chars for c in token):
        banned_token_ids.append(token_id)

print(f"Total Vocab Size : {len(vocab):,}")
print(f"Banned Tokens    : {len(banned_token_ids):,}")
print(f"Allowed Tokens   : {len(vocab) - len(banned_token_ids):,}")

class EnglishOnlyLogitsProcessor(LogitsProcessor):
    def __init__(self, banned_ids, vocab_size):
        # Create a boolean mask once
        self.banned_mask = torch.zeros(vocab_size, dtype=torch.bool)
        self.banned_mask[banned_ids] = True

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        # The scores tensor has shape [batch_size, vocab_size]
        # We need to ensure our mask matches the vocab_size of the scores tensor
        # Sometimes models pad the vocab size to a multiple of 64 or 128 for performance
        
        current_vocab_size = scores.shape[-1]
        
        # If the mask is not on the right device, move it
        if self.banned_mask.device != scores.device:
            self.banned_mask = self.banned_mask.to(scores.device)
            
        # If the model's output layer is larger than the tokenizer's vocab size,
        # we need to pad our mask to match the scores tensor shape.
        # We ban all tokens beyond the tokenizer's known vocab as well.
        if current_vocab_size > len(self.banned_mask):
            padding_size = current_vocab_size - len(self.banned_mask)
            padding = torch.ones(padding_size, dtype=torch.bool, device=scores.device)
            active_mask = torch.cat([self.banned_mask, padding])
        else:
            active_mask = self.banned_mask[:current_vocab_size]
            
        # Apply the mask: set banned token logits to -inf
        # scores is [batch_size, vocab_size], active_mask is [vocab_size]
        # Broadcasting handles the batch dimension automatically
        scores[:, active_mask] = -float('inf')
        
        return scores

# Create the processor list to pass to model.generate()
# We pass the tokenizer's vocab size, but the processor will adapt to the model's actual output size
english_logits_processor = LogitsProcessorList([
    EnglishOnlyLogitsProcessor(banned_token_ids, len(vocab))
])

print("✅ English-only LogitsProcessor created.")

Scanning vocabulary for non-English tokens...
Total Vocab Size : 262,144
Banned Tokens    : 111,608
Allowed Tokens   : 150,536
✅ English-only LogitsProcessor created.


# SimpleChat

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. INTERACTIVE CHAT LOOP (CLI-Style) - v3 Robust Threading
# ══════════════════════════════════════════════════════════════════════════════

class SimpleChat:
    def __init__(self, system_prompt=None):
        self.history = []
        if system_prompt:
             self.history.append({"role": "user", "content": system_prompt})
             self.history.append({"role": "model", "content": "Understood. I am ready."})
            
    def chat_loop(self):
        print("\n💬 GEMMA 3 1B-IT CHAT (Type 'exit' to stop, 'clear' to reset)")
        print("────────────────────────────────────────────────────────")
        
        while True:
            try:
                user_input = input("\n👤 YOU: ").strip()
                if user_input.lower() in ["exit", "quit"]:
                    print("👋 Exiting chat...")
                    break
                if user_input.lower() == "clear":
                    self.history = []
                    print("🧹 Memory cleared.")
                    continue
                if not user_input:
                    continue
            except EOFError:
                break
                
            self.history.append({"role": "user", "content": user_input})
            
            try:
                prompt = tokenizer.apply_chat_template(
                    self.history, 
                    tokenize=False, 
                    add_generation_prompt=True
                )
            except Exception as e:
                print(f"⚠️ Template Error: {e}")
                prompt = f"<start_of_turn>user\n{user_input}<end_of_turn>\n<start_of_turn>model\n"
            
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            print(f"  (Context window: {inputs.input_ids.shape[1]} tokens)")

            print("🤖 MODEL: ", end="", flush=True)
            
            # REMOVED TIMEOUT: Let it wait as long as CPU needs for prefill
            streamer = TextIteratorStreamer(
                tokenizer, 
                timeout=None,    
                skip_prompt=True, 
                skip_special_tokens=True 
            )
            
            terminators = [
                tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids("<end_of_turn>")
            ]
            
            generation_kwargs = dict(
                inputs,
                streamer=streamer,
                max_new_tokens=512,
                do_sample=False,
                eos_token_id=terminators,
                pad_token_id=tokenizer.eos_token_id
            )
            
            # Wrapper to catch errors inside the thread
            def generate_safely():
                try:
                    model.generate(**generation_kwargs)
                except Exception as e:
                    print(f"\n❌ GENERATION THREAD CRASHED: {e}")

            thread = Thread(target=generate_safely)
            thread.start()
            
            full_response = ""
            try:
                for new_text in streamer:
                    if "<end_of_turn>" in new_text:
                        new_text = new_text.replace("<end_of_turn>", "")
                        print(new_text, end="", flush=True)
                        full_response += new_text
                        break
                    print(new_text, end="", flush=True)
                    full_response += new_text
            except Exception as e:
                print(f"\n⚠️ Stream Error: {e}")
                
            print() 
            self.history.append({"role": "model", "content": full_response})

# ─── Launch Chat ───
chat_session = SimpleChat(
    system_prompt="You are a helpful and concise AI assistant. Answer directly."
)



# UI_V9

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════

# GEMMA CHAT v12 — LUXE COPILOT WORKSPACE

# - Premium dark workspace with prominent IndiGo branding

# - Model / Persona / System Prompt selectors

# - Collapsible sections, fluid panel toggles, dynamic chat bubbles

# ══════════════════════════════════════════════════════════════════════════════



import os

import re

from threading import Thread



from PyQt5.QtCore import Qt, QSize, QThread, pyqtSignal, QTimer, QPropertyAnimation, QEasingCurve

from PyQt5.QtGui import QFont, QFontDatabase, QIcon, QPixmap, QTextOption, QColor

from PyQt5.QtWidgets import (

    QApplication, QMainWindow, QWidget, QFrame, QLabel, QPushButton, QLineEdit,

    QTextBrowser, QTextEdit, QListWidget, QListWidgetItem, QFileDialog, QMessageBox,

    QHBoxLayout, QVBoxLayout, QSizePolicy, QScrollArea, QStackedWidget,

    QGraphicsOpacityEffect, QGraphicsDropShadowEffect, QComboBox

    )



from transformers import TextIteratorStreamer



# ── THEME ─────────────────────────────────────────────────────────────────────

CLR_BG = "#070B16"

CLR_SIDEBAR = "#0B1328"

CLR_MAIN = "#0D172D"

CLR_PANEL = "#121F3B"

CLR_PANEL_2 = "#1A2950"

CLR_HOVER = "#23345F"

CLR_BORDER = "#2D3F6A"

CLR_TEXT = "#E7EEFF"

CLR_MUTED = "#9BB1DA"

CLR_ACCENT = "#4A87FF"

CLR_ACCENT_2 = "#6AA8FF"

CLR_USER = "#2E63D9"

CLR_MODEL = "#1B2A4D"

LOGO_PATH = r"C:\Users\divyam.tewary\OneDrive - InterGlobe Aviation Limited\Desktop\IndiGo_logo.png"



MODEL_CHOICES = [

    "Gemma 3 1B-IT (Loaded)",

    "Gemma 3 4B-IT (Design Placeholder)",

    "Gemma 3 12B-IT (Design Placeholder)",

]



PERSONA_CHOICES = {

    "Balanced Assistant": "Be accurate, concise, and practical.",

    "Coder Pro": "You are an elite coding copilot. Return production-ready code with concise rationale.",

    "Analyst": "You are a rigorous analyst. Structure findings clearly with assumptions and trade-offs.",

    "Creative Partner": "You are imaginative and polished while staying grounded in user goals.",

}



SYSTEM_PROMPTS = {

    "Default": "You are a helpful and concise AI assistant. Format clearly using headings and bullet points where useful.",

    "Strict Technical": "You are a strict technical assistant. Keep precision high, avoid fluff, and state assumptions.",

    "Executive Summary": "Respond in executive-ready format with clear sections, key points, and action items.",

    "Tutor Mode": "Teach clearly with examples, step-by-step logic, and short checkpoints.",

}



def _preferred_font():

    families = set(QFontDatabase().families())

    for name in ["Segoe UI Variable", "Inter", "Segoe UI", "Arial"]:

        if name in families:

            return QFont(name, 10)

    return QFont("Sans Serif", 10)



class HoverButton(QPushButton):

    def __init__(self, text, click_handler=None, min_h=36, compact=False, accent=False):

        super().__init__(text)

        self.setCursor(Qt.PointingHandCursor)

        self.setMinimumHeight(min_h)

        pad = "6px 10px" if compact else "8px 12px"

        if accent:

            self.setStyleSheet(

                f"""

                QPushButton {{

                    background: qlineargradient(x1:0,y1:0,x2:1,y2:1, stop:0 {CLR_ACCENT}, stop:1 {CLR_ACCENT_2});

                    color: white;

                    border: 0;

                    border-radius: 11px;

                    padding: {pad};

                    font-weight: 700;

                }}

                QPushButton:hover {{ background: {CLR_ACCENT_2}; }}

                QPushButton:pressed {{ background: {CLR_ACCENT}; }}

                """

            )

        else:

            self.setStyleSheet(

                f"""

                QPushButton {{

                    background-color: {CLR_PANEL};

                    color: {CLR_TEXT};

                    border: 1px solid {CLR_BORDER};

                    border-radius: 10px;

                    padding: {pad};

                    text-align: left;

                    font-weight: 600;

                }}

                QPushButton:hover {{ background-color: {CLR_HOVER}; border: 1px solid {CLR_MUTED}; }}

                QPushButton:pressed {{ background-color: {CLR_PANEL_2}; }}

                """

            )

        if click_handler:

            self.clicked.connect(click_handler)



class ChatBubble(QFrame):

    def __init__(self, role, text=""):

        super().__init__()

        self.role = role

        self.text = text

        self.setObjectName("bubble")



        bubble_grad = (

            "qlineargradient(x1:0,y1:0,x2:1,y2:1, stop:0 #2D66DE, stop:1 #1E4FBF)"

            if role == "user"

            else "qlineargradient(x1:0,y1:0,x2:1,y2:1, stop:0 #16243F, stop:1 #22355C)"

        )



        self.setStyleSheet(

            f"""

            QFrame#bubble {{

                background: {bubble_grad};

                border: 1px solid {CLR_BORDER};

                border-radius: 14px;

            }}

            QLabel {{ color: {CLR_TEXT}; }}

            QTextBrowser {{

                background: transparent;

                border: none;

                color: {CLR_TEXT};

                font-size: 14px;

            }}

            """

        )



        shadow = QGraphicsDropShadowEffect(self)

        shadow.setBlurRadius(22)

        shadow.setOffset(0, 4)

        shadow.setColor(QColor(0, 0, 0, 110))

        self.setGraphicsEffect(shadow)

        self.setSizePolicy(QSizePolicy.Preferred, QSizePolicy.Minimum)



        lay = QVBoxLayout(self)

        lay.setContentsMargins(12, 8, 12, 10)

        lay.setSpacing(6)



        role_lbl = QLabel("You" if role == "user" else "IndiGo AI")

        role_lbl.setStyleSheet(f"color: {CLR_MUTED}; font-weight: 700;")

        lay.addWidget(role_lbl)



        self.body = QTextBrowser()

        self.body.setOpenExternalLinks(True)

        self.body.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Minimum)

        self.body.setMaximumHeight(620)

        self.body.setVerticalScrollBarPolicy(Qt.ScrollBarAlwaysOff)

        self.body.setHorizontalScrollBarPolicy(Qt.ScrollBarAlwaysOff)

        wrap = self.body.document().defaultTextOption()

        wrap.setWrapMode(QTextOption.WrapAtWordBoundaryOrAnywhere)

        self.body.document().setDefaultTextOption(wrap)

        self.body.setMarkdown(self._to_markdown(text))

        lay.addWidget(self.body)



    def _to_markdown(self, raw):

        cleaned = raw.replace("\r\n", "\n")

        cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip()

        return cleaned if cleaned else " "



    def update_text(self, text):

        self.text = text

        self.body.setMarkdown(self._to_markdown(text))

        self.body.document().adjustSize()

        h = int(self.body.document().size().height()) + 26

        self.body.setMinimumHeight(min(max(h, 46), 600))



class GenerationWorker(QThread):

    chunk = pyqtSignal(str)

    finished_ok = pyqtSignal()

    failed = pyqtSignal(str)



    def __init__(self, mdl, tok, prompt_history):

        super().__init__()

        self.model = mdl

        self.tok = tok

        self.prompt_history = prompt_history



    def run(self):

        try:

            prompt = self.tok.apply_chat_template(

                self.prompt_history, tokenize=False, add_generation_prompt=True

            )

            inputs = self.tok(prompt, return_tensors="pt").to(self.model.device)

            streamer = TextIteratorStreamer(

                self.tok, timeout=None, skip_prompt=True, skip_special_tokens=True

            )

            terminators = [self.tok.eos_token_id, self.tok.convert_tokens_to_ids("<end_of_turn>")]



            def _generate():

                self.model.generate(

                    **inputs,

                    streamer=streamer,

                    max_new_tokens=512,

                    do_sample=False,

                    eos_token_id=terminators,

                    pad_token_id=self.tok.eos_token_id,

                )



            t = Thread(target=_generate, daemon=True)

            t.start()



            for token in streamer:

                token = token.replace("<end_of_turn>", "")

                if token:

                    self.chunk.emit(token)



            t.join()

            self.finished_ok.emit()

        except Exception as e:

            self.failed.emit(str(e))



class PremiumGemmaWindow(QMainWindow):

    LEFT_W = 300

    RIGHT_W = 360



    def __init__(self, mdl, tok):

        super().__init__()

        self.model = mdl

        self.tok = tok



        self.worker = None

        self.current_reply = ""

        self.current_bubble = None



        self.sessions = []

        self.current_session_index = -1

        self.files_content = {}



        self.left_expanded = True

        self.right_expanded = True



        self.setWindowTitle("IndiGo AI Workspace")

        self.resize(1540, 950)

        self.setStyleSheet(f"background-color: {CLR_BG}; color: {CLR_TEXT};")

        QApplication.instance().setFont(_preferred_font())



        self.stack = QStackedWidget()

        self.setCentralWidget(self.stack)



        self._build_intro_page()

        self._build_workspace_page()



        self.stack.addWidget(self.intro_page)

        self.stack.addWidget(self.workspace_page)

        self.stack.setCurrentWidget(self.intro_page)



        self.statusBar().setStyleSheet(f"background:{CLR_SIDEBAR}; color:{CLR_MUTED}; border-top:1px solid {CLR_BORDER};")

        self.statusBar().showMessage("Ready")

        self._apply_windows_dark_titlebar()



        self._create_new_session(initial=True)

        QTimer.singleShot(800, self.start_intro_transition)



    # ── Intro ─────────────────────────────────────────────────────────────────

    def _build_intro_page(self):

        self.intro_page = QWidget()

        self.intro_page.setStyleSheet(

            "background: qlineargradient(x1:0,y1:0,x2:1,y2:1, stop:0 #060B16, stop:1 #102040);"

        )



        lay = QVBoxLayout(self.intro_page)

        lay.setContentsMargins(80, 80, 80, 80)

        lay.setSpacing(16)

        lay.addStretch(1)



        logo = QLabel()

        logo.setAlignment(Qt.AlignCenter)

        if os.path.exists(LOGO_PATH):

            pix = QPixmap(LOGO_PATH)

            logo.setPixmap(pix.scaled(240, 78, Qt.KeepAspectRatio, Qt.SmoothTransformation))

        else:

            logo.setText("INDIGO")

            logo.setStyleSheet(f"font-size:36px; font-weight:900; color:{CLR_TEXT};")



        title = QLabel("Premium AI Copilot Workspace")

        title.setAlignment(Qt.AlignCenter)

        title.setStyleSheet(f"font-size: 42px; font-weight: 800; color: {CLR_TEXT};")



        subtitle = QLabel("Model-aware chat with persona and system controls.")

        subtitle.setAlignment(Qt.AlignCenter)

        subtitle.setStyleSheet(f"font-size: 18px; color: {CLR_MUTED};")



        start_btn = HoverButton("Launch Workspace", self.start_intro_transition, min_h=46, accent=True)

        start_btn.setFixedWidth(220)



        btn_wrap = QHBoxLayout()

        btn_wrap.addStretch(1)

        btn_wrap.addWidget(start_btn)

        btn_wrap.addStretch(1)



        lay.addWidget(logo)

        lay.addWidget(title)

        lay.addWidget(subtitle)

        lay.addLayout(btn_wrap)

        lay.addStretch(1)



    def start_intro_transition(self):

        if self.stack.currentWidget() is self.workspace_page:

            return

        effect = QGraphicsOpacityEffect(self.intro_page)

        self.intro_page.setGraphicsEffect(effect)

        self.fade_anim = QPropertyAnimation(effect, b"opacity")

        self.fade_anim.setDuration(420)

        self.fade_anim.setStartValue(1.0)

        self.fade_anim.setEndValue(0.0)

        self.fade_anim.setEasingCurve(QEasingCurve.InOutQuad)

        self.fade_anim.finished.connect(self._show_workspace)

        self.fade_anim.start()



    def _show_workspace(self):

        self.stack.setCurrentWidget(self.workspace_page)

        self.statusBar().showMessage("Workspace ready", 1600)



    # ── Workspace ─────────────────────────────────────────────────────────────

    def _build_workspace_page(self):

        self.workspace_page = QWidget()

        self.workspace_page.setStyleSheet(

            "background: qlineargradient(x1:0,y1:0,x2:1,y2:1, stop:0 #070B16, stop:1 #0E1B35);"

        )

        root = QHBoxLayout(self.workspace_page)

        root.setContentsMargins(0, 0, 0, 0)

        root.setSpacing(0)



        self._build_left_panel()

        self._build_center_panel()

        self._build_right_panel()



        root.addWidget(self.left_panel)

        root.addWidget(self.center_panel, 1)

        root.addWidget(self.right_panel)



    def _build_logo_widget(self):

        w = QWidget()

        lay = QHBoxLayout(w)

        lay.setContentsMargins(4, 4, 4, 4)

        lay.setSpacing(10)



        icon = QLabel()

        icon.setFixedSize(120, 38)

        if os.path.exists(LOGO_PATH):

            pix = QPixmap(LOGO_PATH)

            icon.setPixmap(pix.scaled(118, 36, Qt.KeepAspectRatio, Qt.SmoothTransformation))

        else:

            icon.setText("INDIGO")

            icon.setStyleSheet(f"font-weight:900; color:{CLR_TEXT};")



        sub = QLabel("AI Command Center")

        sub.setStyleSheet(f"font-size:12px; font-weight:700; color:{CLR_MUTED};")



        lay.addWidget(icon)

        lay.addWidget(sub, 1)

        return w



    def _build_section_header(self, title, on_toggle):

        btn = QPushButton(f"▾  {title}")

        btn.setCursor(Qt.PointingHandCursor)

        btn.setStyleSheet(

            f"QPushButton {{ background: transparent; color:{CLR_MUTED}; border:none; text-align:left; font-weight:800; padding:4px 2px; }}"

        )

        btn.clicked.connect(on_toggle)

        return btn



    def _build_left_panel(self):

        self.left_panel = QFrame()

        self.left_panel.setMinimumWidth(0)

        self.left_panel.setMaximumWidth(self.LEFT_W)

        self.left_panel.setStyleSheet(

            f"background: qlineargradient(x1:0,y1:0,x2:0,y2:1, stop:0 {CLR_SIDEBAR}, stop:1 #0D1630); border-right:1px solid {CLR_BORDER};"

        )



        lay = QVBoxLayout(self.left_panel)

        lay.setContentsMargins(12, 12, 12, 12)

        lay.setSpacing(10)



        top_row = QHBoxLayout()

        self.btn_left_toggle = HoverButton("☰", self.on_toggle_left_panel, min_h=34, compact=True)

        self.btn_left_toggle.setFixedWidth(44)

        self.btn_home = HoverButton("Home", self.on_home_clicked, min_h=34, compact=True)

        if os.path.exists(LOGO_PATH):

            self.btn_home.setText("")

            self.btn_home.setIcon(QIcon(LOGO_PATH))

            self.btn_home.setIconSize(QSize(100, 28))

        top_row.addWidget(self.btn_left_toggle)

        top_row.addWidget(self.btn_home, 1)



        self.brand_card = QFrame()

        self.brand_card.setStyleSheet(

            f"QFrame {{ background:{CLR_PANEL}; border:1px solid {CLR_BORDER}; border-radius:14px; }}"

        )

        bc = QVBoxLayout(self.brand_card)

        bc.setContentsMargins(12, 10, 12, 10)

        bc.addWidget(self._build_logo_widget())



        self.btn_new_chat = HoverButton("＋ New Conversation", self.on_new_chat_clicked, min_h=36, accent=True)



        self.section_sessions_btn = self._build_section_header("Conversations", self.on_toggle_sessions_section)

        self.sessions_box = QWidget()

        sessions_lay = QVBoxLayout(self.sessions_box)

        sessions_lay.setContentsMargins(0, 0, 0, 0)

        sessions_lay.setSpacing(6)

        self.session_list = QListWidget()

        self.session_list.setStyleSheet(

            f"""

            QListWidget {{ background:{CLR_MAIN}; color:{CLR_TEXT}; border:1px solid {CLR_BORDER}; border-radius:10px; padding:4px; }}

            QListWidget::item {{ padding:8px; border-radius:8px; }}

            QListWidget::item:hover {{ background:{CLR_PANEL}; }}

            QListWidget::item:selected {{ background:{CLR_HOVER}; }}

            """

        )

        self.session_list.itemClicked.connect(self.on_session_selected)

        sessions_lay.addWidget(self.session_list)



        lay.addLayout(top_row)

        lay.addWidget(self.brand_card)

        lay.addWidget(self.btn_new_chat)

        lay.addWidget(self.section_sessions_btn)

        lay.addWidget(self.sessions_box, 1)



    def _build_center_panel(self):

        self.center_panel = QFrame()

        self.center_panel.setStyleSheet("background: transparent;")

        lay = QVBoxLayout(self.center_panel)

        lay.setContentsMargins(14, 12, 14, 12)

        lay.setSpacing(10)



        header = QFrame()

        header.setStyleSheet(

            f"QFrame {{ background:{CLR_PANEL}; border:1px solid {CLR_BORDER}; border-radius:14px; }}"

        )

        h = QHBoxLayout(header)

        h.setContentsMargins(10, 8, 10, 8)

        h.setSpacing(8)



        self.btn_left_toggle_header = HoverButton("☰", self.on_toggle_left_panel, min_h=34, compact=True)

        self.btn_left_toggle_header.setFixedWidth(44)



        title_wrap = QVBoxLayout()

        title_wrap.setContentsMargins(0, 0, 0, 0)

        title_wrap.setSpacing(0)

        self.title_lbl = QLabel("New Conversation")

        self.title_lbl.setStyleSheet(f"font-size:19px; font-weight:800; color:{CLR_TEXT};")

        self.subtitle_lbl = QLabel("Luxe Workspace")

        self.subtitle_lbl.setStyleSheet(f"font-size:12px; font-weight:600; color:{CLR_MUTED};")

        title_wrap.addWidget(self.title_lbl)

        title_wrap.addWidget(self.subtitle_lbl)



        self.cmb_model = QComboBox()

        self.cmb_model.addItems(MODEL_CHOICES)

        self.cmb_model.currentIndexChanged.connect(self.on_model_changed)

        self.cmb_model.setStyleSheet(self._combo_style())



        self.cmb_persona = QComboBox()

        self.cmb_persona.addItems(list(PERSONA_CHOICES.keys()))

        self.cmb_persona.setStyleSheet(self._combo_style())



        self.cmb_system_prompt = QComboBox()

        self.cmb_system_prompt.addItems(list(SYSTEM_PROMPTS.keys()))

        self.cmb_system_prompt.setStyleSheet(self._combo_style())



        self.btn_toggle_right = HoverButton("Context", self.on_toggle_right_panel, min_h=34, compact=True)

        self.btn_settings = HoverButton("Settings", self.on_settings_clicked, min_h=34, compact=True)



        h.addWidget(self.btn_left_toggle_header)

        h.addLayout(title_wrap)

        h.addStretch(1)

        h.addWidget(self.cmb_model)

        h.addWidget(self.cmb_persona)

        h.addWidget(self.cmb_system_prompt)

        h.addWidget(self.btn_toggle_right)

        h.addWidget(self.btn_settings)



        self.chat_scroll = QScrollArea()

        self.chat_scroll.setWidgetResizable(True)

        self.chat_scroll.setStyleSheet(

            f"QScrollArea {{ border:1px solid {CLR_BORDER}; border-radius:14px; background:{CLR_MAIN}; }}"

        )

        self.chat_host = QWidget()

        self.chat_vbox = QVBoxLayout(self.chat_host)

        self.chat_vbox.setContentsMargins(14, 14, 14, 14)

        self.chat_vbox.setSpacing(12)

        self.chat_vbox.addStretch(1)

        self.chat_scroll.setWidget(self.chat_host)



        input_wrap = QFrame()

        input_wrap.setStyleSheet(

            f"QFrame {{ background:{CLR_PANEL}; border:1px solid {CLR_BORDER}; border-radius:14px; }}"

        )

        i = QHBoxLayout(input_wrap)

        i.setContentsMargins(10, 8, 10, 8)

        i.setSpacing(8)



        self.input_box = QLineEdit()

        self.input_box.setPlaceholderText("Ask anything — code, analysis, planning, docs...")

        self.input_box.setStyleSheet(

            f"QLineEdit {{ background:transparent; border:none; color:{CLR_TEXT}; font-size:14px; }}"

        )

        self.input_box.returnPressed.connect(self.on_send_clicked)



        self.btn_upload = HoverButton("Upload", self.on_upload_clicked, min_h=34, compact=True)

        self.btn_send = HoverButton("Send", self.on_send_clicked, min_h=34, compact=True, accent=True)

        self.btn_send.setFixedWidth(92)



        i.addWidget(self.input_box, 1)

        i.addWidget(self.btn_upload)

        i.addWidget(self.btn_send)



        lay.addWidget(header)

        lay.addWidget(self.chat_scroll, 1)

        lay.addWidget(input_wrap)



    def _build_right_panel(self):

        self.right_panel = QFrame()

        self.right_panel.setMinimumWidth(0)

        self.right_panel.setMaximumWidth(self.RIGHT_W)

        self.right_panel.setStyleSheet(

            f"background: qlineargradient(x1:0,y1:0,x2:0,y2:1, stop:0 #0C1530, stop:1 #101F3E); border-left:1px solid {CLR_BORDER};"

        )



        lay = QVBoxLayout(self.right_panel)

        lay.setContentsMargins(12, 12, 12, 12)

        lay.setSpacing(10)



        row = QHBoxLayout()

        lbl = QLabel("Context & Controls")

        lbl.setStyleSheet(f"font-size:14px; font-weight:800; color:{CLR_TEXT};")

        self.btn_right_toggle = HoverButton("⇤", self.on_toggle_right_panel, min_h=34, compact=True)

        self.btn_right_toggle.setFixedWidth(44)

        row.addWidget(lbl)

        row.addStretch(1)

        row.addWidget(self.btn_right_toggle)



        self.section_files_btn = self._build_section_header("Uploaded Files", self.on_toggle_files_section)

        self.files_box = QWidget()

        files_lay = QVBoxLayout(self.files_box)

        files_lay.setContentsMargins(0, 0, 0, 0)

        files_lay.setSpacing(6)

        self.file_list = QListWidget()

        self.file_list.setStyleSheet(

            f"""

            QListWidget {{ background:{CLR_MAIN}; color:{CLR_TEXT}; border:1px solid {CLR_BORDER}; border-radius:10px; padding:4px; }}

            QListWidget::item {{ padding:8px; border-radius:8px; }}

            QListWidget::item:hover {{ background:{CLR_PANEL}; }}

            QListWidget::item:selected {{ background:{CLR_HOVER}; }}

            """

        )

        self.file_list.itemClicked.connect(self.on_file_selected)

        self.file_preview = QTextEdit()

        self.file_preview.setReadOnly(True)

        self.file_preview.setPlaceholderText("File preview appears here")

        self.file_preview.setStyleSheet(

            f"QTextEdit {{ background:{CLR_MAIN}; color:{CLR_TEXT}; border:1px solid {CLR_BORDER}; border-radius:10px; padding:8px; }}"

        )

        files_lay.addWidget(self.file_list)

        files_lay.addWidget(self.file_preview)



        self.section_controls_btn = self._build_section_header("Runtime Controls", self.on_toggle_controls_section)

        self.controls_box = QWidget()

        controls_lay = QVBoxLayout(self.controls_box)

        controls_lay.setContentsMargins(0, 0, 0, 0)

        controls_lay.setSpacing(8)

        self.info_model = QLabel("Model: Gemma 3 1B-IT")

        self.info_model.setStyleSheet(f"color:{CLR_MUTED};")

        self.info_persona = QLabel("Persona: Balanced Assistant")

        self.info_persona.setStyleSheet(f"color:{CLR_MUTED};")

        self.info_prompt = QLabel("System: Default")

        self.info_prompt.setStyleSheet(f"color:{CLR_MUTED};")

        controls_lay.addWidget(self.info_model)

        controls_lay.addWidget(self.info_persona)

        controls_lay.addWidget(self.info_prompt)



        lay.addLayout(row)

        lay.addWidget(self.section_files_btn)

        lay.addWidget(self.files_box, 1)

        lay.addWidget(self.section_controls_btn)

        lay.addWidget(self.controls_box)



    # ── Interactions ───────────────────────────────────────────────────────────

    def _combo_style(self):

        return (

            f"""

            QComboBox {{

                background:{CLR_MAIN};

                color:{CLR_TEXT};

                border:1px solid {CLR_BORDER};

                border-radius:10px;

                padding:6px 10px;

                min-width:160px;

                font-weight:600;

            }}

            QComboBox QAbstractItemView {{

                background:{CLR_MAIN};

                color:{CLR_TEXT};

                border:1px solid {CLR_BORDER};

                selection-background-color:{CLR_HOVER};

            }}

            """

        )



    def on_home_clicked(self):

        self.stack.setCurrentWidget(self.intro_page)

        self.statusBar().showMessage("Returned to home", 1300)

        QTimer.singleShot(200, self.start_intro_transition)



    def on_settings_clicked(self):

        QMessageBox.information(

            self,

            "Workspace Settings",

            "UI Theme: Luxe Dark\nStreaming: Enabled\nPanels: Animated\nTip: Use selectors in top bar for persona and prompt control.",

        )



    def on_model_changed(self, idx):

        selected = self.cmb_model.currentText()

        if idx != 0:

            self.statusBar().showMessage(f"{selected} not loaded yet. Using current local model.", 2200)

            self.cmb_model.blockSignals(True)

            self.cmb_model.setCurrentIndex(0)

            self.cmb_model.blockSignals(False)

            selected = self.cmb_model.currentText()

        self.info_model.setText(f"Model: {selected}")



    def on_new_chat_clicked(self):

        self._create_new_session()



    def on_toggle_left_panel(self):

        self._animate_panel(self.left_panel, self.LEFT_W, expand=not self.left_expanded)

        self.left_expanded = not self.left_expanded

        self.btn_left_toggle.setText("☰" if self.left_expanded else "⇥")

        self.btn_left_toggle_header.setText("☰" if self.left_expanded else "⇥")



    def on_toggle_right_panel(self):

        self._animate_panel(self.right_panel, self.RIGHT_W, expand=not self.right_expanded)

        self.right_expanded = not self.right_expanded

        self.btn_right_toggle.setText("⇤" if self.right_expanded else "⇥")



    def on_toggle_sessions_section(self):

        show = not self.sessions_box.isVisible()

        self.sessions_box.setVisible(show)

        self.section_sessions_btn.setText(("▾  " if show else "▸  ") + "Conversations")



    def on_toggle_files_section(self):

        show = not self.files_box.isVisible()

        self.files_box.setVisible(show)

        self.section_files_btn.setText(("▾  " if show else "▸  ") + "Uploaded Files")



    def on_toggle_controls_section(self):

        show = not self.controls_box.isVisible()

        self.controls_box.setVisible(show)

        self.section_controls_btn.setText(("▾  " if show else "▸  ") + "Runtime Controls")



    def on_upload_clicked(self):

        paths, _ = QFileDialog.getOpenFileNames(self, "Upload context files", "", "Text Files (*.txt *.md *.py *.json *.csv);;All Files (*)")

        if not paths:

            return



        added = 0

        for path in paths:

            try:

                with open(path, "r", encoding="utf-8", errors="replace") as f:

                    content = f.read()

            except Exception as e:

                QMessageBox.warning(self, "Read error", f"Could not read {path}\n{e}")

                continue

            name = os.path.basename(path)

            self.files_content[name] = content

            added += 1



        self._refresh_file_list()

        self.statusBar().showMessage(f"Added {added} file(s)", 2000)



    def on_file_selected(self, item):

        name = item.text()

        self.file_preview.setPlainText(self.files_content.get(name, ""))



    def on_session_selected(self, item):

        idx = item.data(Qt.UserRole)

        if idx is None or idx < 0 or idx >= len(self.sessions):

            return

        self.current_session_index = idx

        self._load_current_session_into_chat()



    def on_send_clicked(self):

        if self.worker and self.worker.isRunning():

            return

        text = self.input_box.text().strip()

        if not text:

            return



        if self.current_session_index < 0:

            self._create_new_session()



        persona_name = self.cmb_persona.currentText()

        prompt_name = self.cmb_system_prompt.currentText()

        persona_prompt = PERSONA_CHOICES.get(persona_name, "")

        system_prompt = SYSTEM_PROMPTS.get(prompt_name, SYSTEM_PROMPTS["Default"])

        self.info_persona.setText(f"Persona: {persona_name}")

        self.info_prompt.setText(f"System: {prompt_name}")



        session = self.sessions[self.current_session_index]

        if len(session["history"]) == 0:

            session["title"] = text[:46] + ("..." if len(text) > 46 else "")

            self._refresh_session_list()

            self.title_lbl.setText(session["title"] or "New Conversation")



        self.input_box.clear()

        self._add_bubble("user", text)

        session["history"].append({"role": "user", "content": text})



        system_seed = [

            {"role": "user", "content": f"{system_prompt}\n\nPersona mode: {persona_prompt}"},

            {"role": "model", "content": "Understood."},

        ]



        ephemeral = list(system_seed)

        ephemeral.extend(session["history"][:-1])



        context = self._build_context_block()

        aug = f"[REF]\n{context}\n[END]\n\nUser Question: {text}" if context else text

        ephemeral.append({"role": "user", "content": aug})



        self.current_reply = ""

        self.current_bubble = self._add_bubble("model", "Thinking...")



        self.worker = GenerationWorker(self.model, self.tok, ephemeral)

        self.worker.chunk.connect(self.on_stream_chunk)

        self.worker.finished_ok.connect(self.on_stream_finished)

        self.worker.failed.connect(self.on_stream_error)

        self.worker.start()



    def on_stream_chunk(self, chunk):

        self.current_reply += chunk

        if self.current_bubble:

            self.current_bubble.update_text(self.current_reply)

            self._update_bubble_widths()

            self._scroll_to_bottom()



    def on_stream_finished(self):

        if self.current_session_index >= 0 and self.current_reply.strip():

            self.sessions[self.current_session_index]["history"].append(

                {"role": "model", "content": self.current_reply.strip()}

            )

        self.current_reply = ""

        self.current_bubble = None



    def on_stream_error(self, err):

        if self.current_bubble:

            self.current_bubble.update_text(f"⚠ Streaming error: {err}")

        self.current_reply = ""

        self.current_bubble = None



    def showEvent(self, event):

        super().showEvent(event)

        self._apply_windows_dark_titlebar()

        self._update_bubble_widths()



    def resizeEvent(self, event):

        super().resizeEvent(event)

        self._update_bubble_widths()



    # ── Helpers ───────────────────────────────────────────────────────────────

    def _animate_panel(self, panel, expanded_width, expand=True):

        end_w = expanded_width if expand else 0

        self._anim = QPropertyAnimation(panel, b"maximumWidth")

        self._anim.setDuration(220)

        self._anim.setStartValue(panel.maximumWidth())

        self._anim.setEndValue(end_w)

        self._anim.setEasingCurve(QEasingCurve.InOutCubic)

        self._anim.start()



    def _apply_windows_dark_titlebar(self):

        if os.name != "nt":

            return

        try:

            import ctypes

            hwnd = int(self.winId())

            value = ctypes.c_int(1)

            ctypes.windll.dwmapi.DwmSetWindowAttribute(

                ctypes.c_void_p(hwnd),

                ctypes.c_uint(20),

                ctypes.byref(value),

                ctypes.sizeof(value),

            )

        except Exception:

            pass



    def _update_bubble_widths(self):

        if not hasattr(self, "chat_scroll"):

            return

        target = max(520, int(self.chat_scroll.viewport().width() * 0.86))

        for bubble in self.chat_host.findChildren(ChatBubble):

            bubble.setMaximumWidth(target)



    def _build_context_block(self):

        blocks = []

        for name in sorted(self.files_content.keys()):

            txt = self.files_content.get(name, "").strip()

            if txt:

                blocks.append(f"--- FILE: {name} ---\n{txt}\n--- END: {name} ---")

        return "\n\n".join(blocks)



    def _create_new_session(self, initial=False):

        self.sessions.append({"title": "New Conversation", "history": []})

        self.current_session_index = len(self.sessions) - 1

        self._refresh_session_list()

        self._clear_chat_view()

        self.title_lbl.setText("New Conversation")

        if not initial:

            self.statusBar().showMessage("Started new conversation", 1500)



    def _refresh_session_list(self):

        self.session_list.clear()

        for idx, session in enumerate(self.sessions):

            title = session.get("title") or f"Conversation {idx + 1}"

            item = QListWidgetItem(title)

            item.setData(Qt.UserRole, idx)

            self.session_list.addItem(item)

        if 0 <= self.current_session_index < self.session_list.count():

            self.session_list.setCurrentRow(self.current_session_index)



    def _refresh_file_list(self):

        self.file_list.clear()

        for name in sorted(self.files_content.keys()):

            self.file_list.addItem(name)



    def _load_current_session_into_chat(self):

        self._clear_chat_view()

        if self.current_session_index < 0:

            return

        session = self.sessions[self.current_session_index]

        self.title_lbl.setText(session.get("title", "Conversation"))

        for msg in session["history"]:

            self._add_bubble(msg.get("role", "model"), msg.get("content", ""))



    def _add_bubble(self, role, text):

        if self.chat_vbox.count() > 0:

            self.chat_vbox.takeAt(self.chat_vbox.count() - 1)



        bubble = ChatBubble(role, text)

        bubble.setMaximumWidth(max(520, int(self.chat_scroll.viewport().width() * 0.86)))

        row = QHBoxLayout()

        row.setContentsMargins(0, 0, 0, 0)



        if role == "user":

            row.addStretch(1)

            row.addWidget(bubble, 0, Qt.AlignRight)

        else:

            row.addWidget(bubble, 0, Qt.AlignLeft)

            row.addStretch(1)



        holder = QWidget()

        holder.setLayout(row)

        self.chat_vbox.addWidget(holder)

        self.chat_vbox.addStretch(1)

        self._scroll_to_bottom()

        return bubble



    def _clear_chat_view(self):

        while self.chat_vbox.count():

            item = self.chat_vbox.takeAt(0)

            widget = item.widget()

            if widget is not None:

                widget.deleteLater()

        self.chat_vbox.addStretch(1)



    def _scroll_to_bottom(self):

        sb = self.chat_scroll.verticalScrollBar()

        sb.setValue(sb.maximum())



def launch_gemma_premium_ui(mdl, tok):

    try:

        from IPython import get_ipython

        ip = get_ipython()

        if ip is not None:

            ip.run_line_magic("gui", "qt")

    except Exception:

        pass



    app = QApplication.instance()

    owns_app = False

    if app is None:

        app = QApplication([])

        owns_app = True



    win = PremiumGemmaWindow(mdl, tok)

    win.show()



    if owns_app:

        app.exec_()



    return win



# ── LAUNCH ────────────────────────────────────────────────────────────────────

try:

    premium_window = launch_gemma_premium_ui(model, tokenizer)

    print("✅ Luxe Premium PyQt5 UI launched with selectors, branding, and dynamic workspace.")

except Exception as e:

    print(f"❌ Failed to launch Premium UI: {e}")


✅ Luxe Premium PyQt5 UI launched with selectors, branding, and dynamic workspace.


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


# V_9 mark 2

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GEMMA CHAT v9 — 3-PANEL WORKSPACE UI (EXACT REPLICA)
#
# NEW FEATURES:
#   • 3-Panel Layout: Left Sidebar, Center Chat, Right Artifact Viewer
#   • Collapsible sections in the sidebar
#   • Custom dark title bar matching the background (Windows 11)
#   • Polished, responsive UI with rounded inputs and modern icons
# ══════════════════════════════════════════════════════════════════════════════

import tkinter as tk
from tkinter import font as tkfont, filedialog, messagebox
import ctypes, queue, threading, os, re, time
from threading import Thread
from transformers import TextIteratorStreamer

try:
    from PIL import Image, ImageTk, ImageEnhance
    _HAS_PIL = True
except ImportError:
    _HAS_PIL = False

# ── PALETTE ───────────────────────────────────────────────────────────────────
BG_SIDE  = "#1A1E22"  # Deep charcoal gray for sidebar and right panel
BG_MAIN  = "#212121"  # Dark gray for center chat panel
BG_RIGHT = "#1A1E22"  # Right panel background
PANEL    = "#2D2D30"
SURFACE  = "#333333"
HOVER    = "#3E3E42"

U_BUB    = "#2F2F2F"  # User bubble
U_BD     = "#404040"
M_BUB    = "#212121"  # Model bubble (blends with background)
M_BD     = "#333333"

BLUE     = "#3B82F6"
GREEN    = "#22C55E"
WHITE    = "#FFFFFF"

T1       = "#F0F0F5"  # Primary text
T2       = "#A1A1AA"  # Secondary text
T3       = "#6B7280"  # Muted text
T4       = "#4B5563"

BD1      = "#333333"  # Borders

# ── HELPERS ───────────────────────────────────────────────────────────────────
_font_cache9 = {}

def _f9(sz, wt="normal"):
    key = (sz, wt)
    if key not in _font_cache9:
        fams = tkfont.families()
        for nm in ("Segoe UI Variable", "Segoe UI", "Arial"):
            if nm in fams:
                _font_cache9[key] = tkfont.Font(family=nm, size=sz, weight=wt)
                return _font_cache9[key]
        _font_cache9[key] = tkfont.Font(size=sz, weight=wt)
    return _font_cache9[key]

def _dark_tb9(hwnd, bg_hex):
    """Sets the Windows title bar color to match the background."""
    try:
        bg_hex = bg_hex.lstrip('#')
        r, g, b = bg_hex[0:2], bg_hex[2:4], bg_hex[4:6]
        color = int(f"00{b}{g}{r}", 16)
        
        # DWMWA_USE_IMMERSIVE_DARK_MODE = 20
        ctypes.windll.dwmapi.DwmSetWindowAttribute(
            hwnd, 20, ctypes.byref(ctypes.c_int(1)), ctypes.sizeof(ctypes.c_int))
        # DWMWA_CAPTION_COLOR = 35
        ctypes.windll.dwmapi.DwmSetWindowAttribute(
            hwnd, 35, ctypes.byref(ctypes.c_int(color)), ctypes.sizeof(ctypes.c_int))
    except Exception:
        pass

def _rr9(x1, y1, x2, y2, r):
    r = max(1, min(r, (x2 - x1) // 2, (y2 - y1) // 2))
    return [
        x1+r, y1,  x2-r, y1,  x2, y1,  x2, y1+r,
        x2, y2-r,  x2, y2,    x2-r, y2, x1+r, y2,
        x1, y2,    x1, y2-r,  x1, y1+r, x1, y1,
        x1+r, y1,
    ]

# ══════════════════════════════════════════════════════════════════════════════
# MARKDOWN PARSER
# ══════════════════════════════════════════════════════════════════════════════
_MD_BOLD = re.compile(r'\*\*(.+?)\*\*')

def _parse_md_line(line: str):
    segs, last = [], 0
    for m in _MD_BOLD.finditer(line):
        if m.start() > last:
            segs.append((line[last:m.start()], False))
        segs.append((m.group(1), True))
        last = m.end()
    if last < len(line):
        segs.append((line[last:], False))
    return segs if segs else [(" ", False)]

def _parse_md(raw: str):
    result = []
    for line in raw.split('\n'):
        result.append(_parse_md_line(line) if line.strip() else [(" ", False)])
    return result

# ══════════════════════════════════════════════════════════════════════════════
# CHAT CANVAS
# ══════════════════════════════════════════════════════════════════════════════
class FutureChatCanvas9(tk.Canvas):
    PX = 30; BPX = 16; BPY = 12; MAXW = 700; RAD = 12; GAP = 20; TOP = 24

    def __init__(self, parent):
        super().__init__(parent, bg=BG_MAIN, highlightthickness=0, bd=0)
        self._fnt     = _f9(11)
        self._fnt_b   = _f9(11, "bold")
        self._fnt_sm  = _f9(9, "bold")
        self._vw, self._vh = 800, 600
        self._bubbles = []
        self._next_y  = self.TOP
        self._typ_ids = None; self._typ_y = 0; self._typ_f = 0; self._typ_on = False

        self.bind("<Configure>", self._on_resize)
        self.bind_all("<MouseWheel>", lambda e: self.yview_scroll(int(-e.delta / 120), "units"))

    def _on_resize(self, e):
        self._vw, self._vh = e.width, e.height

    def _refresh_sr(self):
        self.configure(scrollregion=(0, 0, self._vw, self._next_y + 60))

    def scroll_end(self):
        def _do():
            self.update_idletasks(); self.yview_moveto(1.0)
        self.after(30, _do)

    def _lh(self): return self._fnt.metrics("linespace")

    def _render_md(self, parsed_lines, x0, y0, fg, max_w):
        ids = []; lh = self._lh(); cy = y0; max_used_w = 0
        for line_segs in parsed_lines:
            cx = x0; line_start_x = x0
            for seg_text, is_bold in line_segs:
                fnt = self._fnt_b if is_bold else self._fnt
                words = seg_text.split(' ')
                for wi, word in enumerate(words):
                    if not word and wi < len(words) - 1:
                        cx += fnt.measure(' '); continue
                    display = (' ' + word) if cx > line_start_x and wi > 0 else word
                    w = fnt.measure(display)
                    if cx + w > x0 + max_w and cx > line_start_x:
                        max_used_w = max(max_used_w, cx - x0)
                        cx = x0; cy += lh; line_start_x = x0
                        display = word; w = fnt.measure(display)
                    tid = self.create_text(cx, cy, text=display, font=fnt, fill=fg, anchor="nw", tags="content")
                    ids.append(tid); cx += w
            max_used_w = max(max_used_w, cx - x0); cy += lh
        return ids, max_used_w, cy - y0

    def add_bubble(self, role, text=""):
        y = self._next_y; bpx, bpy = self.BPX, self.BPY
        
        is_user = role == "user"
        fill = U_BUB if is_user else M_BUB
        bd   = U_BD  if is_user else M_BD
        fg   = WHITE if is_user else T1

        lbl = "You" if is_user else "Gemma"
        lfg = T2
        lx, la = (self._vw - self.PX, "ne") if is_user else (self.PX, "nw")
        lid = self.create_text(lx, y, text=lbl, font=self._fnt_sm, fill=lfg, anchor=la, tags="content")
        y += 20

        parsed = _parse_md(text or " ") if not is_user else [[(text or " ", False)]]
        txt_x = self.PX + bpx
        txt_ids, tw, th = self._render_md(parsed, txt_x, y + bpy, fg, self.MAXW)

        bw = max(tw + bpx * 2, 40); bh = max(th + bpy * 2, 30)
        rx1 = (self._vw - self.PX - bw) if is_user else self.PX
        ry1, rx2, ry2 = y, rx1 + bw, y + bh

        dx = (rx1 + bpx) - txt_x
        if dx:
            for tid in txt_ids:
                ox, oy = self.coords(tid); self.coords(tid, ox + dx, oy)

        pid = self.create_polygon(_rr9(rx1, ry1, rx2, ry2, self.RAD), smooth=True, fill=fill, outline=bd, width=1, tags="content")
        for tid in txt_ids: self.tag_raise(tid, pid)

        self._bubbles.append(dict(role=role, poly=pid, text_ids=txt_ids, label=lid, y1=ry1, y2=ry2, raw_text=text or " "))
        self._next_y = ry2 + self.GAP
        self._refresh_sr(); self.scroll_end()
        return len(self._bubbles) - 1

    def update_bubble(self, idx, text):
        b = self._bubbles[idx]; bpx, bpy = self.BPX, self.BPY
        for tid in b["text_ids"]: self.delete(tid)

        parsed = _parse_md(text) if b["role"] == "model" else [[(text, False)]]
        txt_x = self.PX + bpx
        txt_ids, tw, th = self._render_md(parsed, txt_x, b["y1"] + bpy, T1, self.MAXW)

        bw = max(tw + bpx * 2, 40); bh = max(th + bpy * 2, 30)
        rx1 = (self._vw - self.PX - bw) if b["role"] == "user" else self.PX
        rx2, ry2 = rx1 + bw, b["y1"] + bh

        dx = (rx1 + bpx) - txt_x
        if dx:
            for tid in txt_ids:
                ox, oy = self.coords(tid); self.coords(tid, ox + dx, oy)

        self.coords(b["poly"], _rr9(rx1, b["y1"], rx2, ry2, self.RAD))
        for tid in txt_ids: self.tag_raise(tid, b["poly"])

        b["text_ids"], b["y2"], b["raw_text"] = txt_ids, ry2, text
        self._next_y = ry2 + self.GAP
        self._refresh_sr(); self.scroll_end()

    def show_typing(self):
        y = self._next_y; self._typ_y = y
        lid = self.create_text(self.PX, y, text="Gemma", font=self._fnt_sm, fill=T2, anchor="nw", tags="typing")
        y += 20; w, h = 60, 36; x1 = self.PX
        bgid = self.create_polygon(_rr9(x1, y, x1 + w, y + h, 12), smooth=True, fill=M_BUB, outline=M_BD, width=1, tags="typing")
        dots = []
        for dx in [15, 30, 45]:
            r = 3; ccx, ccy = x1 + dx, y + 18
            oid = self.create_oval(ccx-r, ccy-r, ccx+r, ccy+r, fill=T3, outline="", tags="typing")
            dots.append((oid, ccx, ccy))
        self._typ_ids = dict(bg=bgid, label=lid, dots=dots)
        self._next_y = y + h + self.GAP
        self._refresh_sr(); self.scroll_end()
        self._typ_f = 0; self._typ_on = True; self._anim_dots()

    def hide_typing(self):
        self._typ_on = False
        if self._typ_ids:
            self.delete("typing"); self._next_y = self._typ_y; self._typ_ids = None

    def _anim_dots(self):
        if not self._typ_on or not self._typ_ids: return
        for i, (oid, ccx, ccy) in enumerate(self._typ_ids["dots"]):
            ph = (self._typ_f - i * 3) % 12
            t = max(0, 1 - abs(ph - 3) / 3)
            r = 3 + 1.5 * t
            col = T1 if t > 0.6 else (T2 if t > 0.3 else T3)
            self.coords(oid, ccx-r, ccy-r, ccx+r, ccy+r)
            self.itemconfigure(oid, fill=col)
        self._typ_f += 1; self.after(80, self._anim_dots)

    def clear_all(self):
        self.delete("content"); self.delete("typing")
        self._bubbles.clear(); self._typ_ids = None; self._typ_on = False
        self._next_y = self.TOP; self._refresh_sr()

# ══════════════════════════════════════════════════════════════════════════════
# GENERATION WORKER
# ══════════════════════════════════════════════════════════════════════════════
_SENTINEL9 = object()

def _gen_worker9(mdl, tok, prompt_hist, q):
    try:
        prompt = tok.apply_chat_template(prompt_hist, tokenize=False, add_generation_prompt=True)
        inp = tok(prompt, return_tensors="pt").to(mdl.device)
        streamer = TextIteratorStreamer(tok, timeout=None, skip_prompt=True, skip_special_tokens=True)
        stops = [tok.eos_token_id, tok.convert_tokens_to_ids("<end_of_turn>")]

        def _go():
            try:
                mdl.generate(**inp, streamer=streamer, max_new_tokens=512, do_sample=False, eos_token_id=stops, pad_token_id=tok.eos_token_id, logits_processor=english_logits_processor)
            except Exception as e: q.put(f"\n⚠ {e}")

        t = Thread(target=_go, daemon=True); t.start()
        for chunk in streamer:
            c = chunk.replace("<end_of_turn>", "")
            if c: q.put(c)
        t.join()
    except Exception as e: q.put(f"\n⚠ {e}")
    finally:
        q.put(_SENTINEL9)

# ══════════════════════════════════════════════════════════════════════════════
# FILE TOGGLE (SMART CONTEXT)
# ══════════════════════════════════════════════════════════════════════════════
class FileToggle9(tk.Frame):
    def __init__(self, parent, fname, on_toggle, on_view, on_remove):
        super().__init__(parent, bg=BG_SIDE)
        self.active = True
        self._tog_cb = on_toggle
        
        self._dot = tk.Canvas(self, width=22, height=22, bg=BG_SIDE, highlightthickness=0, cursor="hand2")
        self._dot.pack(side="left", padx=(8, 4), pady=4)
        self._draw_dot()
        self._dot.bind("<Button-1>", self._toggle)
        
        display_name = fname if len(fname) < 18 else fname[:15] + "..."
        self.name_lbl = tk.Label(self, text=display_name, font=_f9(9), bg=BG_SIDE, fg=T1, anchor="w", cursor="hand2")
        self.name_lbl.pack(side="left", fill="x", expand=True, pady=4)
        self.name_lbl.bind("<Button-1>", lambda _: on_view())
        
        self.rm_lbl = tk.Label(self, text="✕", font=_f9(9), bg=BG_SIDE, fg=T4, cursor="hand2", padx=5)
        self.rm_lbl.pack(side="right", pady=4)
        self.rm_lbl.bind("<Button-1>", lambda _: on_remove())
        
        self.bind("<Enter>", lambda _: self._hover(True))
        self.bind("<Leave>", lambda _: self._hover(False))
        for child in self.winfo_children():
            child.bind("<Enter>", lambda _: self._hover(True), add="+")
            child.bind("<Leave>", lambda _: self._hover(False), add="+")

    def _hover(self, is_hover):
        color = HOVER if is_hover else BG_SIDE
        self.configure(bg=color)
        self._dot.configure(bg=color)
        self.name_lbl.configure(bg=color)
        self.rm_lbl.configure(bg=color)

    def _toggle(self, _=None):
        self.active = not self.active
        self._draw_dot()
        self._tog_cb(self.active)

    def _draw_dot(self):
        c = self._dot; c.delete("all")
        if self.active:
            c.create_oval(3, 3, 19, 19, fill=GREEN, outline="")
            c.create_text(11, 11, text="✓", fill=WHITE, font=_f9(7, "bold"))
        else:
            c.create_oval(3, 3, 19, 19, fill="", outline=T4, width=1.5)

# ══════════════════════════════════════════════════════════════════════════════
# MAIN WINDOW
# ══════════════════════════════════════════════════════════════════════════════
class GemmaV9(tk.Tk):
    def __init__(self, mdl, tok):
        super().__init__()
        self.model, self.tok = mdl, tok
        self.history = []
        self._gen = False; self._bub_idx = None; self._cur = ""; self._q = queue.Queue()
        
        self.title("IndiGo Chat Workspace")
        self.geometry("1400x900")
        self.minsize(1000, 600)
        self.configure(bg=BG_MAIN)
        
        def apply_dark_titlebar():
            self.update()
            hwnd = ctypes.windll.user32.GetParent(self.winfo_id())
            if not hwnd: hwnd = self.winfo_id()
            _dark_tb9(hwnd, BG_MAIN)
            
        self.after(120, apply_dark_titlebar)

        self._build()
        self.after(40, self._poll)

    def _build(self):
        # 1. Left Sidebar
        self._build_left_sidebar()
        
        # 2. Right Panel (Artifacts) - Pack this first so it takes right side
        self._build_right_panel()
        
        # 3. Center Panel (Chat)
        self._build_center_panel()

    def _build_left_sidebar(self):
        self.left_frame = tk.Frame(self, bg=BG_SIDE, width=260)
        self.left_frame.pack(side="left", fill="y")
        self.left_frame.pack_propagate(False)
        
        self.sidebar_labels = []
        self.sidebar_expanded = True
        
        # Top items
        top_menu = tk.Frame(self.left_frame, bg=BG_SIDE)
        top_menu.pack(fill="x", pady=10)
        
        self._nav_btn(top_menu, "🏠", "Home")
        self._nav_btn(top_menu, "＋", "New Chat", is_new=True)
        self._nav_btn(top_menu, "👁️", "Uploaded Files Viewer", action=self._toggle_right_panel)
        self._nav_btn(top_menu, "⚙️", "Settings")
        
        # Mid - Context Files
        mid_menu = tk.Frame(self.left_frame, bg=BG_SIDE)
        mid_menu.pack(fill="both", expand=True, pady=10)
        
        self.ctx_open = tk.BooleanVar(value=True)
        self._nav_btn(mid_menu, "📂", "Smart Context Files", action=self._toggle_ctx)
        
        self.ctx_content = tk.Frame(mid_menu, bg=BG_SIDE)
        self.ctx_content.pack(fill="both", expand=True, padx=10, pady=5)
        
        up_btn = tk.Label(self.ctx_content, text="＋ Upload .txt", font=_f9(9, "bold"), bg=SURFACE, fg=T1, cursor="hand2", pady=6)
        up_btn.pack(fill="x", pady=5)
        up_btn.bind("<Button-1>", lambda _: self._upload_context_file())
        
        self.files_container = tk.Frame(self.ctx_content, bg=BG_SIDE)
        self.files_container.pack(fill="both", expand=True)
        
        self._files = {}
        self._cards = {}
        
        # Bottom items
        bot_menu = tk.Frame(self.left_frame, bg=BG_SIDE)
        bot_menu.pack(fill="x", side="bottom", pady=10)
        
        _, self.collapse_icon_lbl = self._nav_btn(bot_menu, "◀", "Collapse", action=self._toggle_sidebar)
        
        user_frame = tk.Frame(bot_menu, bg=BG_SIDE)
        user_frame.pack(fill="x", padx=10, pady=5)
        tk.Label(user_frame, text="👤", bg=BG_SIDE, fg=T1, font=_f9(14)).pack(side="left")
        uf_text = tk.Frame(user_frame, bg=BG_SIDE)
        uf_text.pack(side="left", padx=5)
        u1 = tk.Label(uf_text, text="divyamtewary", bg=BG_SIDE, fg=T1, font=_f9(9, "bold"))
        u1.pack(anchor="w")
        u2 = tk.Label(uf_text, text="Copilot Pro", bg=BG_SIDE, fg=T3, font=_f9(8))
        u2.pack(anchor="w")
        self.sidebar_labels.extend([(u1, "divyamtewary"), (u2, "Copilot Pro")])

    def _nav_btn(self, parent, icon, text, badge=None, is_new=False, action=None):
        f = tk.Frame(parent, bg=BG_SIDE, cursor="hand2")
        f.pack(fill="x", padx=10, pady=2)
        icon_lbl = tk.Label(f, text=icon, bg=BG_SIDE, fg=T1, font=_f9(11))
        icon_lbl.pack(side="left", padx=5, pady=6)
        text_lbl = tk.Label(f, text=text, bg=BG_SIDE, fg=T1, font=_f9(10))
        text_lbl.pack(side="left", pady=6)
        self.sidebar_labels.append((text_lbl, text))
        
        if badge:
            badge_lbl = tk.Label(f, text=badge, bg=GREEN, fg=BG_SIDE, font=_f9(7, "bold"), padx=4, pady=1)
            badge_lbl.pack(side="left", padx=5)
            self.sidebar_labels.append((badge_lbl, badge))
            
        f.bind("<Enter>", lambda e: self._hover(f, True))
        f.bind("<Leave>", lambda e: self._hover(f, False))
        
        def _click(e):
            if is_new: self._new_chat()
            elif action: action()
            
        f.bind("<Button-1>", _click)
        for child in f.winfo_children():
            child.bind("<Enter>", lambda e: self._hover(f, True))
            child.bind("<Leave>", lambda e: self._hover(f, False))
            child.bind("<Button-1>", _click)
            
        return f, icon_lbl

    def _hover(self, frame, is_hover):
        color = HOVER if is_hover else BG_SIDE
        frame.configure(bg=color)
        for child in frame.winfo_children():
            if child.cget("bg") != GREEN: # Don't change badge color
                child.configure(bg=color)

    def _toggle_sidebar(self):
        target = 60 if self.sidebar_expanded else 260
        step = -20 if self.sidebar_expanded else 20
        
        if self.sidebar_expanded:
            for w, _ in self.sidebar_labels: w.configure(text="")
            self.ctx_content.pack_forget()
            self.collapse_icon_lbl.configure(text="▶")
            self.sidebar_expanded = False
        else:
            self.collapse_icon_lbl.configure(text="◀")
            self.sidebar_expanded = True
            
        self._animate_sidebar(self.left_frame.winfo_width(), target, step)

    def _animate_sidebar(self, current, target, step):
        next_w = current + step
        if (step > 0 and next_w >= target) or (step < 0 and next_w <= target):
            self.left_frame.configure(width=target)
            if self.sidebar_expanded:
                for w, txt in self.sidebar_labels: w.configure(text=txt)
                if self.ctx_open.get():
                    self.ctx_content.pack(fill="both", expand=True, padx=10, pady=5)
        else:
            self.left_frame.configure(width=next_w)
            self.after(10, lambda: self._animate_sidebar(next_w, target, step))

    def _toggle_ctx(self):
        if not self.sidebar_expanded:
            self._toggle_sidebar()
            self.ctx_open.set(True)
            self.ctx_content.pack(fill="both", expand=True, padx=10, pady=5)
            return
            
        if self.ctx_open.get():
            self.ctx_content.pack_forget()
            self.ctx_open.set(False)
        else:
            self.ctx_content.pack(fill="both", expand=True, padx=10, pady=5)
            self.ctx_open.set(True)

    def _build_right_panel(self):
        self.right_frame = tk.Frame(self, bg=BG_RIGHT, width=450)
        self.right_frame.pack(side="right", fill="y")
        self.right_frame.pack_propagate(False)
        tk.Frame(self, bg=BD1, width=1).pack(side="right", fill="y") # Divider
        
        # Top Bar Controls
        top_bar = tk.Frame(self.right_frame, bg=BG_RIGHT, height=40)
        top_bar.pack(fill="x")
        top_bar.pack_propagate(False)
        
        tk.Label(top_bar, text="📄 Viewer", bg=BG_RIGHT, fg=T1, font=_f9(9)).pack(side="left", padx=15)
        
        close_btn = tk.Label(top_bar, text="✕", bg=BG_RIGHT, fg=T2, font=_f9(11), cursor="hand2")
        close_btn.pack(side="right", padx=10)
        close_btn.bind("<Button-1>", lambda e: self._toggle_right_panel())
        
        max_btn = tk.Label(top_bar, text="🗖", bg=BG_RIGHT, fg=T2, font=_f9(11), cursor="hand2")
        max_btn.pack(side="right", padx=5)
        
        # Tabs Bar
        self.right_tabs_frame = tk.Frame(self.right_frame, bg=BG_RIGHT, height=30)
        self.right_tabs_frame.pack(fill="x", padx=10, pady=(5, 0))
        
        # Content Displayed (Artifact Viewer)
        self.right_content = tk.Frame(self.right_frame, bg=BG_RIGHT)
        self.right_content.pack(fill="both", expand=True, padx=10, pady=10)
        
        self.open_tabs = {}
        self.active_tab = None
        
        self.empty_lbl = tk.Label(self.right_content, text="Select a file to view its contents here.", bg=BG_RIGHT, fg=T3, font=_f9(10))
        self.empty_lbl.pack(expand=True)
        
        self.right_panel_visible = True

    def _add_tab(self, fn):
        if fn in self.open_tabs: return
        
        display_name = fn if len(fn) < 15 else fn[:12] + "..."
        btn = tk.Label(self.right_tabs_frame, text=display_name, bg=SURFACE, fg=T2, font=_f9(9), padx=10, pady=4, cursor="hand2")
        btn.pack(side="left", padx=(0, 2))
        btn.bind("<Button-1>", lambda e, f=fn: self._switch_tab(f))
        
        txt = tk.Text(self.right_content, bg=BG_RIGHT, fg=T1, font=_f9(10), wrap="word", relief="flat", bd=0, insertbackground=WHITE)
        txt.insert("1.0", self._files.get(fn, ""))
        txt.config(state="disabled")
        
        self.open_tabs[fn] = {"btn": btn, "text": txt}
        self._switch_tab(fn)

    def _remove_tab(self, fn):
        if fn not in self.open_tabs: return
        tab = self.open_tabs.pop(fn)
        tab["btn"].destroy()
        tab["text"].destroy()
        
        if self.active_tab == fn:
            self.active_tab = None
            if self.open_tabs:
                self._switch_tab(list(self.open_tabs.keys())[-1])
            else:
                self.empty_lbl.pack(expand=True)

    def _switch_tab(self, fn):
        if fn not in self.open_tabs: return
        
        if self.active_tab and self.active_tab in self.open_tabs:
            self.open_tabs[self.active_tab]["btn"].configure(bg=SURFACE, fg=T2)
            self.open_tabs[self.active_tab]["text"].pack_forget()
            
        self.empty_lbl.pack_forget()
        
        self.active_tab = fn
        self.open_tabs[fn]["btn"].configure(bg=HOVER, fg=T1)
        self.open_tabs[fn]["text"].pack(fill="both", expand=True)
        
        if not self.right_panel_visible:
            self._toggle_right_panel()

    def _toggle_right_panel(self):
        if self.right_panel_visible:
            self.right_frame.pack_forget()
            self.right_panel_visible = False
        else:
            self.right_frame.pack(side="right", fill="y")
            self.right_panel_visible = True

    def _build_center_panel(self):
        self.center_frame = tk.Frame(self, bg=BG_MAIN)
        self.center_frame.pack(side="left", fill="both", expand=True)
        
        # Top Bar (Global Header)
        top_bar = tk.Frame(self.center_frame, bg=BG_MAIN, height=50)
        top_bar.pack(fill="x")
        top_bar.pack_propagate(False)
        
        # Load IndiGo Logo
        logo_path = r"C:\Users\divyam.tewary\OneDrive - InterGlobe Aviation Limited\Desktop\IndiGo_logo.png"
        try:
            if _HAS_PIL:
                img = Image.open(logo_path)
                resample_filter = getattr(Image, "Resampling", Image).LANCZOS
                aspect = img.width / img.height
                new_w = int(24 * aspect)
                img = img.resize((new_w, 24), resample_filter)
                self._indigo_logo = ImageTk.PhotoImage(img)
                logo_lbl = tk.Label(top_bar, image=self._indigo_logo, bg=BG_MAIN, bd=0)
                logo_lbl.pack(side="left", padx=(20, 10), pady=13)
            else:
                tk.Label(top_bar, text="IndiGo", bg=BG_MAIN, fg=BLUE, font=_f9(14, "bold")).pack(side="left", padx=20)
        except Exception:
            tk.Label(top_bar, text="IndiGo", bg=BG_MAIN, fg=BLUE, font=_f9(14, "bold")).pack(side="left", padx=20)
        
        self.chat_title_var = tk.StringVar(value="New Chat")
        tk.Label(top_bar, textvariable=self.chat_title_var, bg=BG_MAIN, fg=T2, font=_f9(11)).pack(side="left", padx=10)
        
        share_btn = tk.Label(top_bar, text="↗ Share", bg=SURFACE, fg=T1, font=_f9(9), cursor="hand2", padx=10, pady=4)
        share_btn.pack(side="right", padx=20, pady=10)
        
        # Chat Area
        cf = tk.Frame(self.center_frame, bg=BG_MAIN)
        cf.pack(fill="both", expand=True)
        self._chat = FutureChatCanvas9(cf)
        sb = tk.Scrollbar(cf, orient="vertical", bg=BG_MAIN, troughcolor=BG_MAIN,
                          activebackground=BD1, width=8, relief="flat", bd=0)
        self._chat.configure(yscrollcommand=sb.set)
        sb.configure(command=self._chat.yview)
        sb.pack(side="right", fill="y")
        self._chat.pack(side="left", fill="both", expand=True)
        
        # Input Area
        self._build_input()

    def _build_input(self):
        input_container = tk.Frame(self.center_frame, bg=BG_MAIN, pady=15)
        input_container.pack(fill="x", side="bottom")
        
        # Rounded input box wrapper
        self.input_bg = tk.Canvas(input_container, bg=BG_MAIN, highlightthickness=0, height=60)
        self.input_bg.pack(fill="x", padx=40)
        
        self.input_bg.bind("<Configure>", self._draw_input_bg)
        
        # Inner frame for input elements
        inner = tk.Frame(self.input_bg, bg=SURFACE)
        self.input_bg.create_window(20, 10, window=inner, anchor="nw", tags="inner", width=1000)
        
        # Plus icon
        plus_btn = tk.Label(inner, text="＋", bg=SURFACE, fg=T2, font=_f9(14), cursor="hand2")
        plus_btn.pack(side="left", padx=10, pady=10)
        plus_btn.bind("<Button-1>", lambda e: self._upload_context_file())
        
        # Text entry
        self._evar = tk.StringVar()
        self._ebox = tk.Entry(inner, textvariable=self._evar, font=_f9(11), bg=SURFACE, fg=T1, insertbackground=WHITE, relief="flat", bd=0)
        self._ebox.pack(side="left", fill="x", expand=True, ipady=10, padx=5)
        self._ebox.bind("<Return>", lambda _: self._send())
        
        self._ph = True
        self._ebox.insert(0, "Ask anything")
        self._ebox.configure(fg=T4)
        self._ebox.bind("<FocusIn>", self._on_focus_in)
        self._ebox.bind("<FocusOut>", self._on_focus_out)
        
        # Model selector
        model_sel = tk.Label(inner, text="IndiGo RM Agent ▾", bg=SURFACE, fg=T2, font=_f9(9), cursor="hand2")
        model_sel.pack(side="right", padx=15, pady=10)

    def _draw_input_bg(self, event=None):
        c = self.input_bg
        c.delete("bg")
        w = c.winfo_width()
        h = 50
        c.create_polygon(_rr9(10, 5, w-10, h+5, 20), smooth=True, fill=SURFACE, outline=BD1, width=1, tags="bg")
        c.tag_lower("bg")
        c.itemconfigure("inner", width=w-40)

    def _on_focus_in(self, _):
        if self._ph:
            self._ebox.delete(0, "end"); self._ebox.configure(fg=T1); self._ph = False

    def _on_focus_out(self, _):
        if not self._evar.get():
            self._ebox.insert(0, "Ask anything")
            self._ebox.configure(fg=T4); self._ph = True

    def _upload_context_file(self):
        path = filedialog.askopenfilename(filetypes=[("Text", "*.txt"), ("All", "*.*")])
        if not path: return
        fn = os.path.basename(path)
        try: txt = open(path, encoding="utf-8", errors="replace").read()
        except: return
        self._files[fn] = txt
        
        c = FileToggle9(self.files_container, fn, 
                        on_toggle=lambda active, f=fn: self._on_file_toggle(f, active), 
                        on_view=lambda f=fn: self._switch_tab(f),
                        on_remove=lambda f=fn: self._rm_file(f))
        c.pack(fill="x", pady=2)
        self._cards[fn] = c
        self._add_tab(fn)
        self._update_token_count()

    def _on_file_toggle(self, fn, active):
        if active:
            self._add_tab(fn)
        else:
            self._remove_tab(fn)
        self._update_token_count()

    def _rm_file(self, fn):
        if fn in self._cards:
            self._cards.pop(fn).destroy()
        self._files.pop(fn, None)
        self._remove_tab(fn)
        self._update_token_count()

    def _update_token_count(self):
        pass

    def context_block(self):
        return "\n\n".join(f"--- FILE: {fn} ---\n{t.strip()}\n--- END: {fn} ---" for fn, t in self._files.items() if self._cards[fn].active)

    def _new_chat(self):
        if self._gen: return
        self.history.clear()
        self._chat.clear_all()
        self.chat_title_var.set("New Chat")

    def _send(self):
        if self._gen or self._ph: return
        text = self._evar.get().strip()
        if not text: return
        
        if len(self.history) == 0:
            title = text if len(text) < 40 else text[:37] + "..."
            self.chat_title_var.set(title)
            
        self._evar.set(""); self._gen = True; self._cur = ""
        
        self.history.append({"role": "user", "content": text})
        self._chat.add_bubble("user", text)
        
        ephemeral = [{"role": "user", "content": "You are a helpful AI assistant."}, {"role": "model", "content": "Understood."}]
        ephemeral.extend(self.history[:-1])
        
        ctx = self.context_block()
        aug = f"[REF]\n{ctx}\n[END]\n\nUser Question: {text}" if ctx else text
        ephemeral.append({"role": "user", "content": aug})
        
        self._chat.show_typing()
        threading.Thread(target=_gen_worker9, args=(self.model, self.tok, ephemeral, self._q), daemon=True).start()

    def _poll(self):
        try:
            while True:
                item = self._q.get_nowait()
                if item is _SENTINEL9: self._done(); break
                if isinstance(item, str):
                    if self._chat._typ_on: self._chat.hide_typing(); self._bub_idx = self._chat.add_bubble("model", "")
                    self._cur += item; self._chat.update_bubble(self._bub_idx, self._cur)
        except queue.Empty: pass
        self.after(25, self._poll)

    def _done(self):
        if self._chat._typ_on: self._chat.hide_typing()
        if self._cur: self.history.append({"role": "model", "content": self._cur})
        self._gen = False; self._bub_idx = None; self._cur = ""
        self._ebox.focus()

# ── LAUNCH ────────────────────────────────────────────────────────────────────
app9 = GemmaV9(model, tokenizer)
app9.mainloop()
